# Create SciQuality validation and test files

Create the fixed future-evaluation datasets without augmentation or balancing.

In [ ]:
from pathlib import Path
import pandas as pd

# Find the repository root.
ROOT = Path.cwd().resolve()
while not (ROOT / "data_labeling").is_dir():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Repository root could not be found.")
    ROOT = ROOT.parent

INPUT = ROOT / "data_labeling/data_labels/sciquality/labels_sciquality_2010_2026_binary.csv"
OUTPUT = ROOT / "data_labeling/data_labels/sciquality/future_evaluation_2024_2026"

In [ ]:
# Load labels and recover image timestamps.
data = pd.read_csv(INPUT)
text = data["label"].str.extract(r"HMI\.m(\d{4}\.\d{2}\.\d{2}_\d{2}\.\d{2}\.\d{2})")[0]
data["timestamp"] = pd.to_datetime(text, format="%Y.%m.%d_%H.%M.%S")
data = data.sort_values("timestamp").reset_index(drop=True)

assert data["label"].is_unique
assert set(data["goes_class"].unique()) <= {0, 1}
print(f"Images loaded: {len(data):,}")

In [ ]:
# Fixed chronological evaluation periods and their separating gaps.
PERIODS = {
    "validation_2024_2025.csv": ("2024-07-01 00:00", "2025-06-30 23:00"),
    "test_2025_2026_july.csv": ("2025-07-02 00:00", "2026-07-31 23:00"),
}
GAPS = {
    "Stage 3 to validation": ("2024-06-29 11:00", "2024-06-30 10:00"),
    "Validation to test": ("2025-07-01 00:00", "2025-07-01 23:00"),
}

datasets = {name: data[data["timestamp"].between(start, end)].copy()
            for name, (start, end) in PERIODS.items()}

In [ ]:
# Verify both complete 24-hour NF-only gaps and display class counts.
for boundary, (start, end) in GAPS.items():
    gap = data[data["timestamp"].between(start, end)]
    expected = pd.date_range(start, end, freq="h")
    assert len(gap) == 24 and gap["goes_class"].sum() == 0
    assert expected.difference(gap["timestamp"]).empty

summary = []
for name, frame in datasets.items():
    fl = int(frame["goes_class"].sum())
    summary.append({"dataset": name, "samples": len(frame),
                    "NF": len(frame) - fl, "FL": fl,
                    "FL_percent": 100 * fl / len(frame)})
pd.DataFrame(summary)

In [ ]:
# Save natural evaluation distributions using the training-pipeline format.
OUTPUT.mkdir(parents=True, exist_ok=True)
for name, frame in datasets.items():
    path = OUTPUT / name
    frame[["label", "goes_class"]].to_csv(path, index=False)
    print(f"Created: {path} ({len(frame):,} images)")